In [ ]:
import pandas as pd
fp = "../data/sba_loans_prepared/sba_loans_risk_good_train.csv"
df = pd.read_csv(fp)
sel_good = df.RiskyNbrh == 0

In [ ]:
df_gb = df[sel_good]

In [ ]:
cols = df_gb.columns.tolist()
cols = [c for c in cols if c not in ["RiskyNbrh", "LoanStatus"]]
borr_columns = ["BorrCity", "BorrState", "BorrZip", "BusinessType"]
cols = [c for c in cols if c in borr_columns]

In [ ]:
df_gb = df_gb[cols]

In [ ]:
X_gb = df_gb.values

In [ ]:
df_gb["BusinessType"].plot.kde()

In [ ]:
from sklearn.manifold import SpectralEmbedding
embedding = SpectralEmbedding(n_components=2, affinity="nearest_neighbors", n_neighbors=3)
X_transformed = embedding.fit_transform(X_gb)

In [ ]:
df_sp = pd.DataFrame(X_transformed)
df_sp.columns = ["emb-"+str(i+1) for i in range(2)]

In [ ]:
import numpy as np
from sklearn.neighbors import KDTree
NUM_NBRS = 5
tree = KDTree(df_sp, leaf_size=2)              
dist, ind = tree.query(df_sp, k=NUM_NBRS)                
df_dist = pd.DataFrame(dist, columns = ["d-" + str(i+1) for i in range(NUM_NBRS)])

In [ ]:
# Sort the DataFrame by 'Age' in descending order
dist_key = "d-" + str(NUM_NBRS)
df_dist = df_dist.sort_values(by=dist_key, ascending=False)

In [ ]:
df_dist["value"] = [i+1 for i in range(df_dist.shape[0])]

In [ ]:
df_dist[dist_key] = df_dist[dist_key]

In [ ]:
df_dist[dist_key].min()

In [ ]:
subset_sel = (df_dist.value <= 500)
dfp = df_dist[subset_sel]

In [ ]:
dfp.shape

In [ ]:
import plotly.express as px
fig = px.scatter(dfp, x="value", y=dist_key)
fig.show()

In [ ]:
from sklearn.cluster import DBSCAN

In [ ]:
clustering = DBSCAN(eps=3e-16, min_samples=NUM_NBRS).fit(df_sp)
clustering.labels_

In [ ]:
num_clusters = np.unique(clustering.labels_)

In [ ]:
num_clusters

In [ ]:
len(clustering.labels_)

In [ ]:
df_sp.shape

In [ ]:
df_sp["cluster"] = clustering.labels_
df_sp["cluster"] = df_sp["cluster"] .astype(str)
remove_noise = (df_sp.cluster == '-1')
df_sp = df_sp[~remove_noise]

In [ ]:
df_sp.cluster.value_counts()

In [ ]:
px.scatter(df_sp, x=df_sp["emb-1"], y=df_sp["emb-2"], color="cluster")